# Shared VOC Localisation Starter

This notebook is designed to run end-to-end on its own in Google Colab, including dependency installation and dataset download from Google Drive links.

## Table of Contents
- [Overview](#overview)
- [Requirements installation](#requirements-installation)
- [Dataset bootstrap (Google Drive links)](#dataset-bootstrap-google-drive-links)
- [Shared VOC data](#shared-voc-data)
- [EDA: class balance and box quality](#eda-class-balance-and-box-quality)
- [Faster R-CNN setup](#faster-r-cnn-setup)
- [Train and checkpoint](#train-and-checkpoint)
- [Predict and export](#predict-and-export)

## Overview

The workflow is intentionally small and repeatable:
1. Install required packages.
2. Download and extract trainval/test VOC archives from Google Drive links.
3. Build train/val/test tables from XML annotations.
4. Run shared EDA diagnostics on class balance and bounding-box quality.
5. Fine-tune a pretrained Faster R-CNN head on VOC classes.
6. Resume training from a checkpoint when available.
7. Export predictions in the common CSV schema used by the team.

## Requirements installation

Run this section first in Google Colab. It installs project dependencies and also installs `gdown`, which is used to pull dataset archives directly from Google Drive links.

If Colab asks for a runtime restart after installation, restart the session before continuing.

In [12]:
from pathlib import Path

requirements_path = Path('requirements.txt')

if requirements_path.exists():
    %pip install -r requirements.txt gdown
else:
    %pip install numpy pandas matplotlib seaborn pillow torch torchvision transformers accelerate ultralytics gdown

  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]
Note: you may need to restart the kernel to use updated packages.


## Dataset bootstrap (Google Drive links)

This section downloads the VOC `trainval` and `test` archives directly from Google Drive and extracts them into the folder structure expected by the rest of the notebook:

- `trainval/VOC2012/...`
- `test/VOC2012/...`

By default it skips download/extraction if the target folders already exist.

In [14]:
from pathlib import Path
import re
import shutil
import zipfile

import gdown

TRAINVAL_LINK = 'https://drive.google.com/file/d/1UPp6cpOdHnfTQmoAkCIIsDR3pMlkg8Yv/view?usp=share_link'
TEST_LINK = 'https://drive.google.com/file/d/1o3ikA7u5hqXP3CI8GeO_qyA1CQBWt10X/view?usp=share_link'

DATA_ROOT = Path.cwd()
ARCHIVE_DIR = DATA_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False


def extract_drive_file_id(link: str) -> str:
    patterns = [r'/file/d/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)']
    for pattern in patterns:
        match = re.search(pattern, link)
        if match:
            return match.group(1)
    raise ValueError(f'Could not parse Google Drive file id from: {link}')



def find_voc2012_dir(root: Path) -> Path | None:
    for path in root.rglob('VOC2012'):
        if path.is_dir():
            return path
    return None



def ensure_archive(link: str, output_zip: Path, force_download: bool = False) -> Path:
    if output_zip.exists() and not force_download:
        return output_zip
    if output_zip.exists() and force_download:
        output_zip.unlink()

    file_id = extract_drive_file_id(link)
    direct_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url=direct_url, output=str(output_zip), quiet=False)

    if not output_zip.exists() or output_zip.stat().st_size == 0:
        raise RuntimeError(f'Failed to download archive: {output_zip.name}')
    return output_zip



def install_voc_split(split_name: str, archive_path: Path, data_root: Path, force_extract: bool = False) -> Path:
    target = data_root / split_name / 'VOC2012'
    if target.exists() and any(target.iterdir()) and not force_extract:
        return target

    tmp_dir = data_root / '_tmp_extract' / split_name
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(tmp_dir)

    voc_dir = find_voc2012_dir(tmp_dir)
    if voc_dir is None:
        raise RuntimeError(f'VOC2012 folder not found inside {archive_path.name}')

    if target.parent.exists() and force_extract:
        shutil.rmtree(target.parent)
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(voc_dir, target)
    shutil.rmtree(tmp_dir)
    return target


trainval_zip = ensure_archive(TRAINVAL_LINK, ARCHIVE_DIR / 'trainval.zip', force_download=FORCE_REDOWNLOAD)
test_zip = ensure_archive(TEST_LINK, ARCHIVE_DIR / 'test.zip', force_download=FORCE_REDOWNLOAD)

trainval_root = install_voc_split('trainval', trainval_zip, DATA_ROOT, force_extract=FORCE_REEXTRACT)
test_root = install_voc_split('test', test_zip, DATA_ROOT, force_extract=FORCE_REEXTRACT)

print('Trainval ready at:', trainval_root)
print('Test ready at:', test_root)

Downloading...
From (original): https://drive.google.com/uc?id=1UPp6cpOdHnfTQmoAkCIIsDR3pMlkg8Yv
From (redirected): https://drive.google.com/uc?id=1UPp6cpOdHnfTQmoAkCIIsDR3pMlkg8Yv&confirm=t&uuid=d6301e4d-7ca1-4fc7-97d4-6a65c6f91370
To: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/archives/trainval.zip
100%|██████████| 1.95G/1.95G [01:33<00:00, 20.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1o3ikA7u5hqXP3CI8GeO_qyA1CQBWt10X
From (redirected): https://drive.google.com/uc?id=1o3ikA7u5hqXP3CI8GeO_qyA1CQBWt10X&confirm=t&uuid=30ad6b5f-f58b-47c0-a99b-aee48063ec0b
To: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/archives/test.zip
100%|██████████| 1.83G/1.83G [01:23<00:00, 22.0MB/s]

Trainval ready at: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/trainval/VOC2012
Test ready at: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/test/VOC2012


In [ ]:
from pathlib import Path
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
print('Seed:', SEED)

Using device: cpu
Seed: 42


In [15]:
PROJECT_ROOT = Path.cwd()
TRAINVAL_ROOT = PROJECT_ROOT / 'trainval' / 'VOC2012'
TEST_ROOT = PROJECT_ROOT / 'test' / 'VOC2012'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
SPLIT_DIR = ARTIFACTS_DIR / 'group_split'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

for path in [TRAINVAL_ROOT, TEST_ROOT]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected VOC folder: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SPLIT_DIR:', SPLIT_DIR)

PROJECT_ROOT: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN
SPLIT_DIR: /Users/michaelkaramichalis/Library/Mobile Documents/com~apple~CloudDocs/MSc Advanced Computer Science/CMT316 - Applications of Machine Learning/Assessment 2/VOC_MAIN/artifacts/group_split


In [ ]:
def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def load_split_ids(split_name: str) -> list[str]:
    # Prefer locked team split files to avoid leakage.
    shared_path = SPLIT_DIR / f'{split_name}_ids.txt'
    shared_ids = read_ids_from_file(shared_path)
    if shared_ids:
        return shared_ids

    # Fallback to VOC default lists if shared split files are missing.
    root = TRAINVAL_ROOT if split_name in {'train', 'val'} else TEST_ROOT
    fallback_path = root / 'ImageSets' / 'Main' / f'{split_name}.txt'
    fallback_ids = read_ids_from_file(fallback_path)
    if fallback_ids:
        return fallback_ids

    # Final fallback for portable Colab runs: derive IDs from available annotations.
    ann_dir = root / 'Annotations'
    if ann_dir.exists():
        return sorted(p.stem for p in ann_dir.glob('*.xml'))
    return []


def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records


def build_split_tables(root_dir: Path, split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = root_dir / 'Annotations'

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': np.nan, 'height': np.nan})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)

    if not objects_df.empty:
        objects_df['box_w'] = (objects_df['xmax'] - objects_df['xmin'] + 1).clip(lower=1)
        objects_df['box_h'] = (objects_df['ymax'] - objects_df['ymin'] + 1).clip(lower=1)
        objects_df['box_area'] = objects_df['box_w'] * objects_df['box_h']
        objects_df['aspect_ratio'] = objects_df['box_w'] / objects_df['box_h']
        image_area = (objects_df['width'] * objects_df['height']).replace(0, np.nan)
        objects_df['norm_area'] = objects_df['box_area'] / image_area

    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
test_ids = load_split_ids('test')

if not train_ids or not val_ids or not test_ids:
    raise RuntimeError('Missing one or more split files. Check VOC ImageSets/Main or annotation folders.')

train_images, train_objects = build_split_tables(TRAINVAL_ROOT, 'train', train_ids)
val_images, val_objects = build_split_tables(TRAINVAL_ROOT, 'val', val_ids)
test_images, test_objects = build_split_tables(TEST_ROOT, 'test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))

display(train_model_df_ignoring_difficult.head())

Split counts: 5717 5823 10991
train-val overlap: 0
train-test overlap: 0
val-test overlap: 0


,image_id,split,cls,class_id,xmin,ymin,xmax,ymax,box_w,box_h,box_area,difficult,truncated,occluded,width,height
0,2008_000008,train,horse,12,53,87,471,420,419,334,139946,0,0,1,500,442
1,2008_000008,train,person,14,158,44,289,167,132,124,16368,0,1,0,500,442
2,2008_000015,train,bottle,4,270,1,378,176,109,176,19184,0,1,1,500,327
3,2008_000015,train,bottle,4,57,1,164,150,108,150,16200,0,1,1,500,327
4,2008_000019,train,dog,11,139,2,372,197,234,196,45864,0,0,0,480,272


## EDA: Class Balance and Box Quality

This section mirrors the shared EDA workflow from the main notebook so diagnostics are consistent before model training: class balance, annotation flags, and bounding-box geometry.

In [ ]:
trainval_images = pd.concat([train_images, val_images], ignore_index=True).drop_duplicates(subset=['image_id'])
trainval_objects = pd.concat([train_objects, val_objects], ignore_index=True)

if trainval_objects.empty:
    raise RuntimeError('No train/val annotations found. Check annotation paths and split files.')

eda_df = trainval_objects.copy()
obj_counts = eda_df.groupby('cls').size().reindex(VOC_CLASSES, fill_value=0).rename('object_count')
img_counts = eda_df.groupby('cls')['image_id'].nunique().reindex(VOC_CLASSES, fill_value=0).rename('image_count')
class_stats = pd.concat([img_counts, obj_counts], axis=1).sort_values('object_count', ascending=False)

class_stats

In [ ]:
plt.figure(figsize=(12, 5))
order = class_stats.index.tolist()
sns.barplot(data=class_stats.reset_index(), x='cls', y='object_count', order=order, color='#2a9d8f')
plt.xticks(rotation=45, ha='right')
plt.title('Object Count per Class (train + val)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=class_stats.reset_index(), x='cls', y='image_count', order=order, color='#e76f51')
plt.xticks(rotation=45, ha='right')
plt.title('Image Count per Class (train + val)')
plt.tight_layout()
plt.show()

In [ ]:
flag_summary = pd.DataFrame({
    'difficult_rate': [eda_df['difficult'].mean()],
    'truncated_rate': [eda_df['truncated'].mean()],
    'occluded_rate': [eda_df['occluded'].mean()],
})

geom_stats = eda_df[['box_w', 'box_h', 'box_area', 'aspect_ratio', 'norm_area']].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

display(flag_summary)
display(geom_stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(eda_df['box_area'], bins=60, ax=axes[0], color='#264653')
axes[0].set_xscale('log')
axes[0].set_title('BBox Area (log scale)')

sns.histplot(eda_df['aspect_ratio'], bins=60, ax=axes[1], color='#f4a261')
axes[1].set_xlim(0, min(10, np.nanpercentile(eda_df['aspect_ratio'], 99)))
axes[1].set_title('Aspect Ratio (w/h)')

sns.histplot(eda_df['norm_area'].dropna(), bins=60, ax=axes[2], color='#e9c46a')
axes[2].set_title('Normalized BBox Area')

plt.tight_layout()
plt.show()

In [7]:
class VOCDetectionDataset(Dataset):
    def __init__(self, root: Path, image_ids: list[str], objects_df: pd.DataFrame, transforms=None):
        self.root = root
        self.image_ids = list(image_ids)
        self.transforms = transforms
        self.objects_by_image = {}

        if not objects_df.empty:
            for image_id, group in objects_df.groupby('image_id'):
                self.objects_by_image[image_id] = group.copy()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.root / 'JPEGImages' / f'{image_id}.jpg'
        image = Image.open(image_path).convert('RGB')

        rows = self.objects_by_image.get(image_id)
        if rows is None or rows.empty:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(rows[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(), dtype=torch.float32)
            labels = torch.tensor(rows['class_id'].to_numpy(), dtype=torch.int64)
            area = torch.tensor(rows['box_area'].to_numpy(), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'area': area,
            'iscrowd': torch.zeros((boxes.shape[0],), dtype=torch.int64),
            'image_id': image_id,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


train_ds = VOCDetectionDataset(TRAINVAL_ROOT, train_ids, train_model_df_ignoring_difficult)
val_ds = VOCDetectionDataset(TRAINVAL_ROOT, val_ids, val_model_df_ignoring_difficult)
test_ds = VOCDetectionDataset(TEST_ROOT, test_ids, test_objects)

print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))

Dataset sizes: 5717 5823 10991


In [8]:
def compute_iou_xyxy(box_a: np.ndarray, box_b: np.ndarray) -> float:
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1 + 1)
    inter_h = max(0.0, inter_y2 - inter_y1 + 1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1 + 1) * max(0.0, ya2 - ya1 + 1)
    area_b = max(0.0, xb2 - xb1 + 1) * max(0.0, yb2 - yb1 + 1)

    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union



def ap_from_pr(precisions: np.ndarray, recalls: np.ndarray) -> float:
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))



def evaluate_map50(pred_df: pd.DataFrame, gt_df: pd.DataFrame, classes: list[str], iou_thresh: float = 0.5) -> dict:
    if pred_df.empty or gt_df.empty:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    per_class_ap = {}
    valid_classes = []

    for cls in classes:
        gt_cls = gt_df[gt_df['cls'] == cls]
        if gt_cls.empty:
            continue

        valid_classes.append(cls)
        pred_cls = pred_df[pred_df['cls'] == cls].sort_values('score', ascending=False)

        gt_by_image = {}
        for image_id, group in gt_cls.groupby('image_id'):
            gt_by_image[image_id] = {
                'boxes': group[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float),
                'matched': np.zeros(len(group), dtype=bool),
            }

        tps = np.zeros(len(pred_cls), dtype=float)
        fps = np.zeros(len(pred_cls), dtype=float)

        for i, row in enumerate(pred_cls.itertuples(index=False)):
            image_state = gt_by_image.get(row.image_id)
            if image_state is None or len(image_state['boxes']) == 0:
                fps[i] = 1.0
                continue

            pred_box = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
            ious = np.array([compute_iou_xyxy(pred_box, gt_box) for gt_box in image_state['boxes']])
            best_idx = int(np.argmax(ious))
            best_iou = ious[best_idx]

            if best_iou >= iou_thresh and not image_state['matched'][best_idx]:
                tps[i] = 1.0
                image_state['matched'][best_idx] = True
            else:
                fps[i] = 1.0

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)
        recalls = cum_tp / max(1, len(gt_cls))
        precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
        per_class_ap[cls] = ap_from_pr(precisions, recalls) if len(pred_cls) > 0 else 0.0

    if not valid_classes:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    map50 = float(np.mean([per_class_ap[cls] for cls in valid_classes]))
    return {'map50': map50, 'per_class_ap': per_class_ap}



def ensure_prediction_schema(df: pd.DataFrame, source_model: str, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out['source_model'] = source_model
    out['split'] = split_name
    for col in PREDICTION_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[PREDICTION_COLUMNS]
    out['class_id'] = out['class_id'].astype('Int64')
    return out

## Faster R-CNN setup

This section builds the pretrained detector, replaces the classification head for the 20 VOC classes plus background, and prepares the shared loaders.

The notebook also stores a lightweight checkpoint so you can stop and resume the same training run without rebuilding everything from scratch.

In [10]:
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor



def pil_to_tensor_transform(image, target):
    return T.ToTensor()(image), target



def build_faster_rcnn_model(num_classes: int) -> torch.nn.Module:
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model



def targets_to_device(targets: list[dict], device: torch.device) -> list[dict]:
    moved_targets = []
    for target in targets:
        moved_targets.append({
            'boxes': target['boxes'].to(device),
            'labels': (target['labels'] + 1).to(device),  # VOC labels are shifted because 0 is background
            'area': target['area'].to(device),
            'iscrowd': target['iscrowd'].to(device),
        })
    return moved_targets



def filter_model_outputs(output: dict, score_threshold: float = 0.05, max_detections: int = 100) -> dict:
    scores = output['scores'].detach().cpu().numpy()
    keep = scores >= score_threshold

    if keep.sum() == 0:
        return {
            'boxes': np.empty((0, 4), dtype=float),
            'scores': np.empty((0,), dtype=float),
            'labels': np.empty((0,), dtype=int),
        }

    boxes = output['boxes'].detach().cpu().numpy()[keep][:max_detections]
    scores = scores[keep][:max_detections]
    labels = output['labels'].detach().cpu().numpy()[keep][:max_detections] - 1

    return {
        'boxes': boxes,
        'scores': scores,
        'labels': labels,
    }



def save_checkpoint(model, optimizer, epoch: int, path: Path) -> None:
    torch.save(
        {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'seed': SEED,
        },
        path,
    )



def load_checkpoint(path: Path, map_location=DEVICE):
    if not path.exists():
        return None
    return torch.load(path, map_location=map_location)


train_ds = VOCDetectionDataset(TRAINVAL_ROOT, train_ids, train_model_df_ignoring_difficult, transforms=pil_to_tensor_transform)
val_ds = VOCDetectionDataset(TRAINVAL_ROOT, val_ids, val_model_df_ignoring_difficult, transforms=pil_to_tensor_transform)
test_ds = VOCDetectionDataset(TEST_ROOT, test_ids, test_objects, transforms=pil_to_tensor_transform)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, collate_fn=collate_fn)

CHECKPOINT_DIR = ARTIFACTS_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / 'fasterrcnn_last.pt'

model = build_faster_rcnn_model(len(VOC_CLASSES) + 1)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
START_EPOCH = 0
resume_state = load_checkpoint(CHECKPOINT_PATH)
if resume_state is not None:
    model.load_state_dict(resume_state['model_state_dict'])
    optimizer.load_state_dict(resume_state['optimizer_state_dict'])
    START_EPOCH = int(resume_state.get('epoch', -1)) + 1
    print(f'Resumed Faster R-CNN from {CHECKPOINT_PATH} at epoch {START_EPOCH}')
else:
    print('No checkpoint found; starting a fresh run.')

print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))

No checkpoint found; starting a fresh run.
Train batches: 2859
Val batches: 2912
Test batches: 5496


In [ ]:
# Minimal fine-tuning loop for the shared VOC split.
EPOCHS = 2
for epoch in range(START_EPOCH, START_EPOCH + EPOCHS):
    model.train()
    running_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = targets_to_device(targets, DEVICE)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())

    save_checkpoint(model, optimizer, epoch, CHECKPOINT_PATH)
    print(f'Epoch {epoch + 1} loss: {running_loss / max(1, len(train_loader)):.4f}')
    print(f'Checkpoint saved to {CHECKPOINT_PATH}')

## Predict and export

Once training completes, this section runs inference on the validation and test loaders, filters low-confidence boxes, and writes the shared CSV prediction files used by the ensemble notebook.

In [ ]:
def predict_to_df(model, loader, split_name: str, score_threshold: float = 0.05, max_detections: int = 100) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for images, targets in loader:
            image_ids = [t['image_id'] for t in targets]
            images_gpu = [img.to(DEVICE) for img in images]
            outputs = model(images_gpu)

            for image_id, out in zip(image_ids, outputs):
                filtered = filter_model_outputs(out, score_threshold=score_threshold, max_detections=max_detections)
                boxes = filtered['boxes']
                scores = filtered['scores']
                labels = filtered['labels']

                for box, score, class_id in zip(boxes, scores, labels):
                    if class_id < 0 or class_id >= len(VOC_CLASSES):
                        continue
                    rows.append({
                        'image_id': image_id,
                        'class_id': int(class_id),
                        'cls': ID_TO_CLASS[int(class_id)],
                        'xmin': float(box[0]),
                        'ymin': float(box[1]),
                        'xmax': float(box[2]),
                        'ymax': float(box[3]),
                        'score': float(score),
                    })

    return ensure_prediction_schema(pd.DataFrame(rows), source_model='fasterrcnn', split_name=split_name)


val_pred_df = predict_to_df(model, val_loader, split_name='val')
test_pred_df = predict_to_df(model, test_loader, split_name='test')

val_path = PREDICTIONS_DIR / 'fasterrcnn_val_predictions.csv'
test_path = PREDICTIONS_DIR / 'fasterrcnn_test_predictions.csv'
val_pred_df.to_csv(val_path, index=False)
test_pred_df.to_csv(test_path, index=False)

metrics = evaluate_map50(val_pred_df, val_model_df_ignoring_difficult, VOC_CLASSES)
print('Faster R-CNN val mAP@0.5:', round(metrics['map50'], 4))
print('Saved:', val_path)
print('Saved:', test_path)